# CAS Exam 5: IncrementalAdditive and sample_weight in Reserving

Dataset used:
- `.../chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

**Learning goals:**
- Understand `cl.IncrementalAdditive`: what makes it *additive* and when to prefer it over chain ladder
- See how `sample_weight` (exposure) is used inside `IncrementalAdditive` to compute zeta factors
- Understand the broader role of `sample_weight` across chainladder estimators (BF, CapeCod, IncrementalAdditive)

## Formula Sheet Reference

### Multiplicative (Chain Ladder) vs Additive (IncrementalAdditive)

| | **Multiplicative (Chain Ladder)** | **Additive (IncrementalAdditive)** |
|---|---|---|
| **Unit of analysis** | Link ratio: $f_{d} = C_{w,d+1} / C_{w,d}$ | Zeta: $\zeta_{d} = \Delta C_{w,d} / \text{Exposure}_w$ |
| **Projection** | Future cumulative = Latest × CDF | Future incremental = Exposure × $\sum_{d>\text{latest}} \zeta_d$ |
| **Key assumption** | Incremental development is proportional to *prior cumulative* | Incremental development is proportional to *exposure* |
| **When it outperforms CL** | Development factors stable across AYs | Incrementals per unit exposure are stable; LDFs near 1.0; thin data |
| **Sample weight role** | Not used (pure development) | Required: divides incremental by exposure to compute $\zeta$ |

### IncrementalAdditive Zeta Factors

$$\zeta_{d} = \frac{\Delta C_{w,d}}{\text{Exposure}_w}$$

The fitted zeta at each development age is the volume-weighted (or simple) average across accident years:

$$\hat{\zeta}_{d} = \frac{\sum_w \Delta C_{w,d}}{\sum_w \text{Exposure}_w}$$

This is the additive analogue of the volume-weighted LDF — numerator is total incremental, denominator is total exposure (not prior cumulative).

### The `trend` Parameter

`IncrementalAdditive(trend=t)` trends each accident year's incrementals to the valuation date before averaging:

$$\zeta_{w,d}^{\text{trended}} = \zeta_{w,d} \times (1 + t)^{\text{valuation date} - \text{origin date}}$$

This adjusts for claim cost inflation across accident years before computing the zeta pattern.

### sample_weight in chainladder Estimators

| Estimator | sample_weight role |
|---|---|
| `Chainladder` | Not used |
| `IncrementalAdditive` | Divides incrementals to produce zeta factors (**required**) |
| `BornhuetterFerguson` | Exposure basis: Expected Ultimate = `apriori × sample_weight` |
| `Benktander` | Same as BF (initial expected ultimate) |
| `CapeCod` | Used-up premium denominator: `sum(sample_weight × % reported)` |
| `ExpectedLoss` | Expected Ultimate = `apriori × sample_weight` |

The `sample_weight` argument is **always an exposure Triangle** — a triangle shaped like the latest diagonal, one row per accident year, holding that AY's exposure (earned premium, policy count, or other volume measure).

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import chainladder as cl


from reservingengine.reserving import build_exposure_triangle

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto.csv'
raw = pd.read_csv(DATA_PATH)

triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims'],
    cumulative=True,
)
paid = triangle['Paid Claims']
paid

## 1. Building the sample_weight (Exposure) Triangle

`sample_weight` must be a Triangle, not a plain Series. The utility `build_exposure_triangle` converts
an AY-indexed exposure Series into a Triangle shaped like the latest diagonal of the claims triangle.

This exposure triangle is reused across `IncrementalAdditive`, `BornhuetterFerguson`, etc.

Here we construct a plausible earned-premium series from the data:
- Back out an earned premium proxy by dividing latest paid by an assumed loss ratio.

In [ ]:
# Back-solve for an exposure (earned premium proxy) from the latest diagonal.
# Assume an a priori loss ratio of 0.75 to infer premium.
latest_paid = paid.latest_diagonal.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0]
ay_years = latest_paid.index.year

# Infer earned premium from latest reported / assumed loss ratio
apriori_lr = 0.75
earned_premium = (latest_paid / apriori_lr).values

exposure_series = pd.Series(earned_premium, index=ay_years, dtype=float)

# Develop the triangle first — IncrementalAdditive operates on a developed triangle
dev = cl.Development(average='volume', n_periods=-1).fit_transform(paid)

# Build the sample_weight triangle from the exposure series
sample_weight = build_exposure_triangle(exposure_series, dev)

print('Exposure (earned premium proxy) by accident year:')
exposure_df = pd.DataFrame({
    'LatestPaid': latest_paid.values,
    'EarnedPremiumProxy': earned_premium,
    'LossRatioProxy': (latest_paid.values / earned_premium),
}, index=ay_years)
exposure_df.index.name = 'AccidentYear'
exposure_df

## 2. Fitting IncrementalAdditive

`IncrementalAdditive` is a **development transformer** — it replaces `cl.Development` in a pipeline.
Its `.fit()` method requires `sample_weight` (the exposure triangle).

After fitting:
- `zeta_` — fitted incremental amount per unit of exposure at each development age
- `ldf_` — implied multiplicative LDFs (derived from zeta for compatibility)
- `tri_zeta` — raw zeta values observed at each (origin, development) cell

The additive projection adds `exposure × zeta_d` for each future development period, not `cumulative × LDF`.

In [ ]:
# IncrementalAdditive replaces cl.Development as the development transformer.
# Note: fit() takes sample_weight here, not transform().
ia = cl.IncrementalAdditive(average='volume', n_periods=-1)
ia.fit(paid, sample_weight=sample_weight)

# The zeta_ triangle: fitted incremental per unit exposure at each development age
zeta_df = ia.zeta_.to_frame()
print('Fitted zeta factors (incremental claims per unit exposure):')
print(zeta_df.to_string())
print()

# Compare the raw per-AY zeta observations to the fitted (averaged) zeta
print('Raw per-AY zeta observations (tri_zeta), for the first two development periods:')
raw_zeta = ia.tri_zeta.to_frame(origin_as_datetime=False)
raw_zeta

## 3. IncrementalAdditive vs Chain Ladder — What Changes

The implied LDFs from `IncrementalAdditive.ldf_` are *not* the same as the LDFs from `cl.Development`.
This is because `IncrementalAdditive` derives LDFs from the additive projection:

$$\text{Implied LDF}_{d} = 1 + \frac{\hat{\zeta}_{d} \times \text{Avg Exposure}}{\text{Avg Cumulative at age } d}$$

The additive method effectively produces **lower LDFs for large-cumulative AYs** and **higher LDFs for small-cumulative AYs** — averaging by exposure rather than by prior cumulative means each AY contributes equally per dollar of exposure, not per dollar of prior claims.

**When this matters:** If older AYs have larger cumulative values (more mature, more claims), volume-weighted chain ladder over-weights them. Additive zeta averaging gives equal weight per exposure unit — which is more natural when the incremental is driven by the *underwriting volume* of the year, not its prior loss history.

In [ ]:
# Compare IncrementalAdditive implied LDFs vs standard Development LDFs
ldf_ia  = ia.ldf_.to_frame().iloc[0]
ldf_dev = cl.Development(average='volume', n_periods=-1).fit(paid).ldf_.to_frame().iloc[0]

ldf_compare = pd.DataFrame({
    'IncrementalAdditive LDF': ldf_ia,
    'Development (volume)   ': ldf_dev,
    'Difference             ': ldf_ia - ldf_dev,
})
print('LDF comparison: IncrementalAdditive vs volume-weighted chain ladder')
ldf_compare

## 4. Reserve Comparison: IncrementalAdditive vs Chain Ladder

Because `IncrementalAdditive` is a transformer, the same `Chainladder()` estimator closes out the reserve —
the difference is entirely in the development pattern. The pipeline replaces `cl.Development` with
`cl.IncrementalAdditive`, and `Chainladder()` projects from there as usual.

In [ ]:
# Pipeline A: traditional Development + Chainladder
dev_transformed  = cl.Development(average='volume', n_periods=-1).fit_transform(paid)
cl_model_dev     = cl.Chainladder().fit(dev_transformed)

# Pipeline B: IncrementalAdditive + Chainladder
# IncrementalAdditive.transform() fills the lower triangle using the fitted zeta pattern
ia_transformed   = ia.transform(paid)
cl_model_ia      = cl.Chainladder().fit(ia_transformed)

ay_index = paid.origin.year
comparison = pd.DataFrame({
    'Latest':        paid.latest_diagonal.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'CL Ultimate':   cl_model_dev.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'IA Ultimate':   cl_model_ia.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'CL IBNR':       cl_model_dev.ibnr_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'IA IBNR':       cl_model_ia.ibnr_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
}, index=ay_index)
comparison.index.name = 'AccidentYear'
comparison['IBNR Diff (IA−CL)'] = comparison['IA IBNR'] - comparison['CL IBNR']
comparison.loc['Total'] = comparison.sum()
comparison

## 5. sample_weight Across Estimators — A Unified View

The `sample_weight` argument appears in multiple chainladder estimators but plays a **different role** in each.
Understanding this distinction is essential for correct implementation.

### IncrementalAdditive: sample_weight as a normalizer

Here, `sample_weight` divides the incremental to produce a zeta factor:

$$\zeta_{w,d} = \frac{\Delta C_{w,d}}{\text{exposure}_w}$$

Without `sample_weight`, there is no way to normalize incrementals across accident years of different sizes.
**sample_weight is required** for `IncrementalAdditive.fit()`.

### BornhuetterFerguson / Benktander: sample_weight as an expected-ultimate scaler

$$\text{Expected Ultimate}_w = \text{apriori} \times \text{exposure}_w$$

The `apriori` parameter (e.g., expected loss ratio) is multiplied by the exposure to get the a priori expected
ultimate for each accident year. Without `sample_weight`, the model cannot compute AY-level expected ultimates.

### CapeCod: sample_weight in the ECR denominator

$$\text{ECR}_{CC} = \frac{\sum_w \text{Latest}_w}{\sum_w \text{exposure}_w \times \% \text{Reported}_w}$$

The exposure (on-level premium) anchors the denominator. A wrong exposure input biases the Cape Cod ECR.

In [ ]:
# Demonstrate sample_weight usage across three estimators using the same exposure triangle.
# All three produce different ultimates because the role of sample_weight differs.

bf_model  = cl.BornhuetterFerguson(apriori=apriori_lr).fit(dev_transformed, sample_weight=sample_weight)
ia_cl     = cl.Chainladder().fit(ia.transform(paid))   # IncrementalAdditive already used sample_weight in fit()
cc_model  = cl.CapeCod(trend=0.0, decay=1.0).fit(dev_transformed, sample_weight=sample_weight)

sw_comparison = pd.DataFrame({
    'CL (no exposure)':    cl_model_dev.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'IncrementalAdditive': cl_model_ia.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'BF (apriori=0.75)':   bf_model.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
    'Cape Cod':            cc_model.ultimate_.to_frame(origin_as_datetime=False, keepdims=True).iloc[:, 0].values,
}, index=ay_index)
sw_comparison.index.name = 'AccidentYear'
sw_comparison.loc['Total'] = sw_comparison.sum()
sw_comparison

In [ ]:
# Bar chart: total IBNR by method to visualize how exposure input shifts estimates
total_ibnr = {
    'Chain Ladder\n(no exposure)':  float(np.nansum(cl_model_dev.ibnr_.values)),
    'IncrementalAdditive\n(exposure-normalized)': float(np.nansum(cl_model_ia.ibnr_.values)),
    'BF\n(exposure × apriori)':     float(np.nansum(bf_model.ibnr_.values)),
    'Cape Cod\n(self-calibrating)': float(np.nansum(cc_model.ibnr_.values)),
}

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['steelblue', 'darkorange', 'seagreen', 'mediumpurple']
bars = ax.bar(list(total_ibnr.keys()), [v / 1e6 for v in total_ibnr.values()], color=colors, alpha=0.85)
for bar, val in zip(bars, total_ibnr.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'${val / 1e6:.1f}M', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Total IBNR ($M)')
ax.set_title('Total IBNR by Method — same data, different use of exposure')
ax.set_ylim(0, max(total_ibnr.values()) / 1e6 * 1.2)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary and When to Use Each

### IncrementalAdditive: Use When

- **Development factors are close to 1.0 (tail):** Multiplicative LDFs near 1.0 are numerically equivalent to additive factors; additive averaging is more stable since small variations in cumulative don't amplify.
- **Thin triangles / sparse data:** With few accident years, additive zeta averaging pools information naturally by exposure rather than by accident-year cumulative size.
- **Incremental losses behave like a pure exposure function:** If each AY's incremental claims at age $d$ are roughly proportional to that AY's earned premium, additive is the right structural assumption.
- **Trend adjustment is needed:** The `trend` parameter corrects for claim cost inflation before averaging — something chain ladder LDFs can't do directly.

### IncrementalAdditive: Avoid When

- Exposure data is unavailable or unreliable (sample_weight is required).
- Development factors are large (early ages): in early development, the multiplicative assumption is typically more accurate because the prior cumulative is a strong predictor.

### The sample_weight Principle (Exam Pattern)

Whenever an exam question asks about a method that uses **exposure** or **premium**, that method needs a `sample_weight`:
- BF, Benktander, CapeCod → `sample_weight` = on-level earned premium by AY
- IncrementalAdditive → `sample_weight` = the same exposure measure used to compute zeta
- Chain Ladder → no `sample_weight`

**Common exam mistake:** Passing `sample_weight` to `Chainladder()` — it is silently ignored. Only pass `sample_weight` to estimators that document its use.

| | Requires `sample_weight`? | What it does |
|---|---|---|
| `Chainladder` | No | — |
| `IncrementalAdditive` | **Yes** (in `.fit()`) | Normalizes incrementals to produce zeta |
| `BornhuetterFerguson` | Yes | Scales apriori to per-AY expected ultimate |
| `Benktander` | Yes | Same as BF (initial iteration uses BF) |
| `CapeCod` | Yes | Provides exposure in used-up premium denominator |
| `ExpectedLoss` | Yes | Same as BF |
